In [ ]:
#!/usr/bin/env python3
"""
Model 5: Innovation — Training Script
=======================================
This is your team's innovation model. Identify a problem in the data that
Models 1-4 don't address, and build a model that solves it.

Requirements:
- Clear value proposition (why does this model matter?)
- Defined success metric (you choose what to optimize)
- Cost-benefit estimate (what's the ROI?)

Use whatever approach fits your problem best — traditional ML, deep learning,
clustering, anomaly detection, time series, etc.
"""
from pathlib import Path
import pandas as pd

PROCESSED_DATA = Path("data/processed/")
SAVED_MODEL_DIR = Path("models/model5_innovation/saved_model/")



In [ ]:
def load_data():
    """Load data for your innovation model."""
    
    #pull the processed data
    filepath = PROCESSED_DATA / "patient_encounters_2023_processed.csv"
    return pd.read_csv(filepath)



In [ ]:
def preprocess(df):
    """Preprocess data for your chosen problem."""
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler

    #drop target
    df = df.copy()
    df = df.dropna(subset=["diabetesMed"])

    #select all columns with potential data leakage and drop them
    med_cols = [
        'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
        'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
        'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
        'miglitol', 'troglitazone', 'tolazamide', 'examide',
        'citoglipton', 'insulin', 'glyburide-metformin',
        'glipizide-metformin', 'glimepiride-pioglitazone',
        'metformin-rosiglitazone', 'metformin-pioglitazone',
        'num_active_meds', 'n_meds_changed', 'n_meds_increased'
    ]

    existing_med_cols = [col for col in med_cols if col in df.columns]
    df = df.drop(columns=existing_med_cols, errors="ignore")

    #set target and features
    y = df["diabetesMed"]
    X = df.drop(columns=["diabetesMed"], errors="ignore")

    #train test split
    X_train, X_val, y_train, y_val = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    #select scaling columns and scale them
    scale_cols = [
        'age',
        'time_in_hospital',
        'num_lab_procedures',
        'num_procedures',
        'num_medications',
        'number_outpatient',
        'number_emergency',
        'number_inpatient',
        'number_diagnoses',
        'complexity_score',
        'max_glu_serum',
        'A1Cresult'
    ]

    scale_cols = [col for col in scale_cols if col in X_train.columns]

    scaler = StandardScaler()

    X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
    X_val[scale_cols] = scaler.transform(X_val[scale_cols])

    return X_train, X_val, y_train, y_val, scaler

In [ ]:
def train_model(X_train, y_train):
    """Train your innovation model."""
    from sklearn.linear_model import LogisticRegression

    #create baseline logistic regression model
    model = LogisticRegression(max_iter=1000, class_weight="balanced")
    model.fit(X_train, y_train)
    return model



In [ ]:
def evaluate_model(model, X_val, y_val):
    """Evaluate with your custom success metric.

    Must include:
    - Your chosen metric (and why you chose it)
    - Baseline comparison (what would a naive approach get?)
    - Business impact estimate
    """
    from sklearn.metrics import accuracy_score, f1_score, classification_report

    y_pred = model.predict(X_val)
    accuracy = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    baseline = max(y_val.mean(), 1 - y_val.mean())

    print("=== Innovation Model Evaluation ===")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Naive baseline accuracy: {baseline:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_val, y_pred))

    print("\nChosen success metric: F1 Score")
    print("Reason: It balances precision and recall for identifying patients likely")
    print("to require diabetes medication.")

    print("\nValue proposition:")
    print("This model helps estimate diabetes medication need using non-medication")
    print("clinical features, which could support early intervention and care planning.")

    print("\nCost-benefit estimate:")
    print("If providers can identify likely medication need earlier, they may improve")
    print("care coordination and reduce delays in treatment decisions.")

In [ ]:

def save_model(model):
    """Save your model to saved_model/.

    Example:
        import joblib
        SAVED_MODEL_DIR.mkdir(parents=True, exist_ok=True)
        joblib.dump(model, SAVED_MODEL_DIR / "model.joblib")
    """
    import joblib

    SAVED_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, SAVED_MODEL_DIR / "model.joblib")
    joblib.dump(scaler, SAVED_MODEL_DIR / "scaler.joblib")



In [ ]:

def main():
    # 1. Load data
    df = load_data()

    # 2. Preprocess
    # X_train, X_val, y_train, y_val = preprocess(df)

    X_train, X_val, y_train, y_val, scaler = preprocess(df)

    # 3. Train
    # model = train_model(X_train, y_train)
    
    model = train_model(X_train, y_train)

    # 4. Evaluate
    # evaluate_model(model, X_val, y_val)

    evaluate_model(model, X_val, y_val)

    # 5. Save
    # save_model(model)

    save_model(model, scaler)

    print("Training complete!")



In [ ]:

if __name__ == "__main__":
    main()
